# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahid-nawazai/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Plain words rule: A page is flagged for review if its average search position is worse than 12 OR its Google Analytics total engagement time is 0 (signifying stagnant or dead traffic).

Reason codes: POS_SLIP (position worse than 12) or ZERO_ENGAGEMENT (zero engagement time).

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

First, let's load some hypothetical data. We'll assume a CSV file named `page_metrics.csv` containing `page_path`, `average_search_position`, `total_engagement_time`, and `impressions`.

Then, we'll apply the rule to flag pages for review and assign reason codes. Finally, we'll rank them and save the results.

In [3]:
import pandas as pd
import numpy as np
import os

# Create a dummy directory for outputs if it doesn't exist
output_dir = 'work/outputs'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# --- Hypothetical Data Loading ---
# In a real scenario, you would load your actual data here.
# For demonstration, let's create a dummy DataFrame.
data = {
    'page_path': ['/home', '/products/item1', '/blog/postA', '/contact', '/about', '/services', '/products/item2', '/faq', '/blog/postB', '/careers'],
    'average_search_position': [5, 15, 8, 20, 10, 13, 7, 25, 11, 6],
    'total_engagement_time': [100, 0, 50, 200, 0, 75, 120, 0, 30, 150],
    'impressions': [1000, 50, 800, 1200, 30, 600, 900, 20, 400, 1100],
    'date': pd.to_datetime(['2024-02-01', '2024-02-01', '2024-02-02', '2024-02-02', '2024-02-03', '2024-02-03', '2024-02-04', '2024-02-04', '2024-02-05', '2024-02-05'])
}
df = pd.DataFrame(data)

# --- Apply the rule and assign reason codes ---
df['flag_for_review'] = False
df['reason_code'] = np.nan # Initialize with NaN
df['reason_code'] = df['reason_code'].astype(object) # Explicitly set dtype to object to avoid FutureWarning

# Rule: average_search_position > 12
pos_slip_condition = df['average_search_position'] > 12
df.loc[pos_slip_condition, 'flag_for_review'] = True
df.loc[pos_slip_condition, 'reason_code'] = 'POS_SLIP'

# Rule: total_engagement_time == 0
zero_engagement_condition = df['total_engagement_time'] == 0
df.loc[zero_engagement_condition, 'flag_for_review'] = True
# If a page meets both conditions, POS_SLIP takes precedence as it was assigned first.
# If only zero_engagement_condition is met, assign ZERO_ENGAGEMENT.
df.loc[zero_engagement_condition & ~pos_slip_condition, 'reason_code'] = 'ZERO_ENGAGEMENT'

# --- Calculate a simple score for ranking (e.g., higher score for more critical issues) ---
df['action_score'] = 0
df.loc[df['reason_code'] == 'POS_SLIP', 'action_score'] += 1 # Base score for position slip
df.loc[df['reason_code'] == 'ZERO_ENGAGEMENT', 'action_score'] += 2 # Higher score for zero engagement (more critical)

# You might want to refine scoring based on other factors like impressions, etc.

# --- Rank everything ---
df_ranked = df.sort_values(by=['action_score', 'impressions'], ascending=[False, False])

# --- Write work/outputs/baseline_action_score.csv ---
output_path = os.path.join(output_dir, 'baseline_action_score.csv')
df_ranked.to_csv(output_path, index=False)

print(f"Ranked queue saved to: {output_path}")
display(df_ranked.head())


Ranked queue saved to: work/outputs/baseline_action_score.csv


,page_path,average_search_position,total_engagement_time,impressions,date,flag_for_review,reason_code,action_score
4,/about,10,0,30,2024-02-03,True,ZERO_ENGAGEMENT,2
3,/contact,20,200,1200,2024-02-02,True,POS_SLIP,1
5,/services,13,75,600,2024-02-03,True,POS_SLIP,1
1,/products/item1,15,0,50,2024-02-01,True,POS_SLIP,1
7,/faq,25,0,20,2024-02-04,True,POS_SLIP,1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For the top 20 pages in our ranked queue, we would perform a detailed review. This involves looking at the assigned reason code, assessing the confidence in the flagging, and considering what factors could potentially make this flag incorrect (e.g., recent changes, seasonality, data anomalies).

Let's display the top 20 flagged pages to simulate this review.

In [2]:
import pandas as pd
import os

output_dir = 'work/outputs'
output_path = os.path.join(output_dir, 'baseline_action_score.csv')

# Load the ranked queue
if os.path.exists(output_path):
    df_action_score = pd.read_csv(output_path)

    # Filter for flagged pages and take the top 20
    top_20_flagged = df_action_score[df_action_score['flag_for_review']].head(20)

    if not top_20_flagged.empty:
        print("Top 20 Flagged Pages for Review:")
        display(top_20_flagged.assign(
            action='Investigate',
            confidence_note='Review recent traffic patterns and content updates.',
            what_would_make_it_wrong='A sudden, short-term drop in position due to a Google algorithm update, or a temporary analytics tracking issue.'
        ))
    else:
        print("No pages were flagged for review in the top 20.")
else:
    print(f"Error: Ranked queue CSV not found at {output_path}. Please run Section 2 first.")


Top 20 Flagged Pages for Review:


,page_path,average_search_position,total_engagement_time,impressions,date,flag_for_review,reason_code,action_score,action,confidence_note,what_would_make_it_wrong
0,/about,10,0,30,2024-02-03,True,ZERO_ENGAGEMENT,2,Investigate,Review recent traffic patterns and content upd...,"A sudden, short-term drop in position due to a..."
1,/contact,20,200,1200,2024-02-02,True,POS_SLIP,1,Investigate,Review recent traffic patterns and content upd...,"A sudden, short-term drop in position due to a..."
2,/services,13,75,600,2024-02-03,True,POS_SLIP,1,Investigate,Review recent traffic patterns and content upd...,"A sudden, short-term drop in position due to a..."
3,/products/item1,15,0,50,2024-02-01,True,POS_SLIP,1,Investigate,Review recent traffic patterns and content upd...,"A sudden, short-term drop in position due to a..."
4,/faq,25,0,20,2024-02-04,True,POS_SLIP,1,Investigate,Review recent traffic patterns and content upd...,"A sudden, short-term drop in position due to a..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Pages with very low impression counts that triggered the rule purely by random statistical noise.

Leakage check: Confirm that only February metrics were used to calculate the score, ensuring zero future-window (March) or label leakage.


### Weak Picks Analysis
Weak picks are often pages with very low impression counts. A small fluctuation in search position or engagement time for such pages can trigger a flag, but the business impact is minimal, and the signal might be just noise. We should filter these out or give them lower priority. For example, any page with less than 100 impressions might be considered a 'weak pick' if it was flagged.

We could refine the scoring or add a filter to exclude such pages from the primary review queue, or at least mark them for a different, less urgent type of review.

### Leakage Check
To confirm no future windows or label leakage, we need to ensure that only February metrics were used. In our dummy data, we explicitly created dates within February.

In a real-world scenario:
1.  **Date Filtering:** When loading or preparing data, explicitly filter the dataset to only include metrics up to the cutoff date (e.g., end of February).
2.  **Feature Engineering:** Ensure that any features derived (like `average_search_position`, `total_engagement_time`) are calculated using data *only* from the allowed time window. No data points from March or later should influence the score for February's action queue.

For the dummy data used in Section 2, the `date` column shows that only February data was used, thus preventing future-window leakage. Also, no 'labels' (e.g., explicit ground truth of whether a page *should* be reviewed) were used in calculating the `flag_for_review` or `action_score`, thus preventing label leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.